## **Applio**
A simple, high-quality voice conversion tool focused on ease of use and performance.

[Support](https://discord.gg/urxFjYmYYh) — [GitHub](https://github.com/IAHispano/Applio) — [Terms of Use](https://github.com/IAHispano/Applio/blob/main/TERMS_OF_USE.md)

<br>

---

<br>

#### **Acknowledgments**

To all external collaborators for their special help in the following areas:
* Hina (Encryption method)
* Poopmaster (Extra section)
* Shirou (UV installer)
* Bruno5430 (AutoBackup code and general notebook maintenance)

#### **Disclaimer**
By using Applio, you agree to comply with ethical and legal standards, respect intellectual property and privacy rights, avoid harmful or prohibited uses, and accept full responsibility for any outcomes, while Applio disclaims liability and reserves the right to amend these terms.

### **Install Applio**
If the runtime restarts, re-run the installation steps.

In [ ]:
# @title Mount Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Setup runtime environment
from IPython.display import clear_output
import codecs

encoded_url = "uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/"
decoded_url = codecs.decode(encoded_url, "rot_13")

repo_name_encoded = "Nccyvb"
repo_name = codecs.decode(repo_name_encoded, "rot_13")

LOGS_PATH = f"/content/{repo_name}/logs"
BACKUPS_PATH = f"/content/drive/MyDrive/{repo_name}Backup"

%cd /content
!git config --global advice.detachedHead false
!git clone {decoded_url} --branch 3.6.0 --single-branch
%cd {repo_name}
clear_output()

# Install older python
!apt update -y
!apt install -y python3.11 python3.11-distutils python3.11-dev portaudio19-dev
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!update-alternatives --set python3 /usr/bin/python3.11
from sys import path
path.append('/usr/local/lib/python3.11/dist-packages')

print("Installing requirements...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel &> /dev/null

!python core.py "prerequisites" --models "True" --pretraineds_hifigan "True"
print("✅ Finished installing requirements!")


In [ ]:
# @title **Start server**
# @markdown  ### Choose a sharing method:
from IPython.display import clear_output

method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "If you selected the 'ngrok' method, obtain your auth token here: https://dashboard.ngrok.com/get-started/your-authtoken" # @param {type:"string"}
tensorboard = True #@param {type: "boolean"}

%cd /content/{repo_name}
clear_output()

if tensorboard:
  %load_ext tensorboard
  %tensorboard --logdir logs --bind_all

match method:
  case 'gradio':
    !python app.py --listen --share --client
  case 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !echo
    !lt --port 6969 & python app.py --listen --client
  case 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f"Ngrok URL: {listener.url()}")
    !python app.py --listen --client

In [ ]:
# ═══════════════════════════════════════════════════════════
# APPLIO TAM KURULUM - TEK CELL (8-10 dakika)
# ═══════════════════════════════════════════════════════════

import os
import sys

# ──────────────────────────────────────────────────────────
# 1/6 - Clone (2 dk)
# ──────────────────────────────────────────────────────────
print("📥 1/6 - Applio clone ediliyor...")
os.chdir('/content')
!rm -rf Applio 2>/dev/null || true
!git clone -q https://github.com/IAHispano/Applio.git

if not os.path.exists('/content/Applio/app.py'):
    print("❌ HATA: Clone başarısız!")
    sys.exit(1)

os.chdir('/content/Applio')
print("✅ Clone tamamlandı")

# ──────────────────────────────────────────────────────────
# 2/6 - Requirements düzeltme
# ──────────────────────────────────────────────────────────
print("\n🔧 2/6 - requirements.txt düzenleniyor...")
!sed -i 's/faiss-cpu==1.7.3/faiss-cpu==1.8.0/g' requirements.txt
!grep -i faiss requirements.txt
print("✅ faiss-cpu 1.8.0'a güncellendi")

# ──────────────────────────────────────────────────────────
# 3/6 - PyTorch (3 dk)
# ──────────────────────────────────────────────────────────
print("\n🔥 3/6 - PyTorch kuruluyor...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print("✅ PyTorch kuruldu")

# ──────────────────────────────────────────────────────────
# 4/6 - Kritik paketler (2 dk)
# ──────────────────────────────────────────────────────────
print("\n📦 4/6 - Temel paketler kuruluyor...")
!pip install -q gradio==4.44.0 huggingface_hub==0.25.2 numpy==1.26.4 faiss-cpu==1.8.0
print("✅ Temel paketler kuruldu")

# ──────────────────────────────────────────────────────────
# 5/6 - Diğer requirements (3 dk)
# ──────────────────────────────────────────────────────────
print("\n📦 5/6 - Diğer paketler kuruluyor...")
!pip install -q -r requirements.txt 2>&1 | tail -20
print("✅ Tüm paketler kuruldu")

# ──────────────────────────────────────────────────────────
# 6/6 - GPU Kontrolü
# ──────────────────────────────────────────────────────────
print("\n🎮 6/6 - GPU kontrol ediliyor...")
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ CUDA: {torch.version.cuda}")
else:
    print("⚠️ GPU YOK - Runtime ayarlarını kontrol et!")

# ──────────────────────────────────────────────────────────
# SONUÇ
# ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("✅ KURULUM TAMAMLANDI!")
print("="*60)
print("\nŞimdi Applio'yu başlat:")
print("!python app.py --share")

